In [5]:
import pandas as pd

# ================== Load All Datasets ==================

df1 = pd.read_csv("../data/raw/reviews.csv") 
df2 = pd.read_csv("../data/raw/hard_reviews.csv")
df3 = pd.read_csv("../data/raw/real_reviews.csv")
df4 = pd.read_csv("../data/raw/real-world.csv")  # synthetic/old

print(len(df1), len(df2), len(df3), len(df4))

5000 203 115 40432


In [6]:
# Label encoding and data cleaning
df1.columns = ["review_text", "label", "rating"]

df2.columns = ["review_text", "label", "rating"]

df3.columns = ["review_text", "label", "rating"]

df4.columns = ["rating", "label", "review_text"]

def clean_df(df):
    df = df.dropna(subset=["review_text"])
    df["review_text"] = df["review_text"].astype(str)
    df = df[df["review_text"].str.strip() != ""]
    df = df[df["review_text"].str.len() > 5]
    return df


In [7]:
# Cleaning

def clean_df(df):
    df = df.dropna(subset=["review_text"])
    df["review_text"] = df["review_text"].astype(str)
    df = df[df["review_text"].str.strip() != ""]
    df = df[df["review_text"].str.len() > 5]
    return df

In [8]:
# Apply cleaning

df1 = clean_df(df1)
df2 = clean_df(df2)
df3 = clean_df(df3)
df4 = clean_df(df4)

In [9]:
# Normalize labels
def normalize_label(x):
    x = str(x).lower()
    if x in ["fake", "1"]:
        return 1
    else:
        return 0

df1["label"] = df1["label"].apply(normalize_label)
df2["label"] = df2["label"].apply(normalize_label)
df3["label"] = df3["label"].apply(normalize_label)
df4["label"] = df4["label"].apply(normalize_label)


In [10]:
# Sampling

df_clean = pd.concat([df1, df2, df3], ignore_index=True)
df_noisy = df4.copy()

df_clean["source"] = "clean"
df_noisy["source"] = "noisy"

print("Clean:", len(df_clean))
print("Noisy:", len(df_noisy))

total_size = 10000

clean_target = int(0.75 * total_size)
noisy_target = total_size - clean_target

df_clean_sample = df_clean.sample(n=clean_target, replace=True, random_state=42)
df_noisy_sample = df_noisy.sample(n=noisy_target, replace=True, random_state=42)

df = pd.concat([df_clean_sample, df_noisy_sample], ignore_index=True)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Final dataset:", len(df))
print(df["label"].value_counts())
print(df["source"].value_counts())

Clean: 5318
Noisy: 40432
Final dataset: 10000
label
1    5691
0    4309
Name: count, dtype: int64
source
clean    7500
noisy    2500
Name: count, dtype: int64


In [11]:
# Combine text

import random

def combine_text(row):
    text = str(row["review_text"])
    rating = str(row["rating"])

    r = random.random()

    if r < 0.5:
        return text
    elif r < 0.8:
        return f"{text} rating {rating}"
    else:
        return f"{text}. Rating: {rating}"

df["combined_text"] = df.apply(combine_text, axis=1)

In [12]:
# Save

df.to_csv("../data/processed/reviews_cleaned.csv", index=False)

In [13]:
# Sampling stats

print("Total samples:", len(df))

print("\nLabel distribution:")
print(df["label"].value_counts())

print("\nSource distribution:")
print(df["source"].value_counts())

print("\nSample rows:")
print(df.sample(5))

Total samples: 10000

Label distribution:
label
1    5691
0    4309
Name: count, dtype: int64

Source distribution:
source
clean    7500
noisy    2500
Name: count, dtype: int64

Sample rows:
                                            review_text  label  rating source  \
9560       not bad. pretty sure its cheaper elsewhere.       1       2  clean   
6890  After using this phone, I feel it does the job...      0       3  clean   
5266  After using this watch, I feel it satisfied wi...      0       4  clean   
7816         VISIT THIS LINK AND WIN EXCITING PRIZES!!!      1       2  clean   
6318  TOP QUALITY GUARANTEED!!! Everyone should buy ...      1       4  clean   

                                          combined_text  
9560  not bad. pretty sure its cheaper elsewhere.  r...  
6890  After using this phone, I feel it does the job...  
5266  After using this watch, I feel it satisfied wi...  
7816         VISIT THIS LINK AND WIN EXCITING PRIZES!!!  
6318  TOP QUALITY GUARANTEED!!! 

In [14]:
# # Add more short fake patterns to further improve

# short_fake_data = [
#     ["SHOCKED!!!", 5, 1],
#     ["WOW!!!", 5, 1],
#     ["OMG!!!", 5, 1],
#     ["UNBELIEVABLE!!!", 5, 1],
#     ["INSANE!!!", 5, 1],
#     ["WHAT IS THIS!!!", 5, 1],
#     ["CANNOT BELIEVE THIS!!!", 5, 1],
#     ["WORST EVER!!!", 1, 1],
#     ["SO BAD!!!", 1, 1],
#     ["TERRIBLE!!!", 1, 1],
# ] * 20  